# ForgeAudio (`audio-forge`) — Remote A100/L4 GPU Worker

This notebook runs stem separation on a **Google Colab A100 / L4 GPU** and exposes a FastAPI endpoint that the local Rust `forge` CLI and `forge-mcp` server call when `FORGE_COLAB_WORKER_URL` is set.

**Implemented:** `POST /api/v1/execute` with `task: separate_stems` runs a **BS-RoFormer** vocal/instrumental model (`model_bs_roformer_ep_317_sdr_12.9755.ckpt` by default, or `payload.model`) through `audio-separator` and returns two stems, `vocals` and `other` (the instrumental). Drums, bass, guitar and keys are not separated here; use the Rust DSP splitter (`FORGE_WORKER_BACKEND=dsp`) when you need them.

**Not implemented:** 6-stem separation, Apollo restoration, YuE2-3B and ACE-Step generation, SheetSage2 / Basic-Pitch transcription, and Seed-VC / RVC conversion. Any other `task` is rejected with HTTP 400.

In [ ]:
!pip install -q fastapi uvicorn[standard] audio-separator[gpu] soundfile librosa numpy pyngrok huggingface_hub torch torchaudio

In [ ]:
import os, json, subprocess
from pathlib import Path
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import torch

app = FastAPI(title="ForgeAudio Colab A100 Worker")

class WorkerRequest(BaseModel):
    task: str
    payload: dict

@app.get("/health")
def health():
    return {
        "ok": True,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "vram_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2) if torch.cuda.is_available() else 0
    }

def _stem_tag(filename):
    # audio-separator names outputs "<input>_(<Stem>)_<model>.wav"; only the last group counts.
    name = Path(filename).name
    start = name.rfind("_(")
    end = name.find(")", start + 2) if start >= 0 else -1
    return "".join(c for c in name[start + 2:end].lower() if c.isalnum()) if end >= 0 else None

STEM_ROLES = {"vocals": {"vocals", "vocal"}, "other": {"instrumental", "instrumentals"}}

@app.post("/api/v1/execute")
def execute(req: WorkerRequest):
    if req.task == "separate_stems":
        # The forge client uploads the audio as base64 and expects base64 WAV stems back,
        # because this machine cannot read paths on the caller's disk.
        import base64, tempfile
        from audio_separator.separator import Separator
        # Each request gets its own directory, removed once the stems are encoded, so a
        # long-lived runtime does not fill its disk with uploads and stems.
        with tempfile.TemporaryDirectory(prefix="forge_") as tmp:
            work = Path(tmp)
            src = work / Path(req.payload.get("filename", "input.wav")).name
            src.write_bytes(base64.b64decode(req.payload["audio_b64"]))
            sep = Separator(output_dir=str(work), output_format="WAV")
            sep.load_model(model_filename=req.payload.get("model", "model_bs_roformer_ep_317_sdr_12.9755.ckpt"))
            stems = {}
            for f in sep.separate(str(src)):
                path = Path(f) if Path(f).is_absolute() else work / f
                for role, tags in STEM_ROLES.items():
                    if role not in stems and _stem_tag(f) in tags:
                        stems[role] = base64.b64encode(path.read_bytes()).decode("ascii")
            if not stems:
                raise HTTPException(status_code=500, detail="separator produced no vocal or instrumental stem")
        return {"ok": True, "task": req.task, "data": {"stems": stems}}
    raise HTTPException(status_code=400, detail=f"Unknown task: {req.task}")